# Benchmark dataloders of Tahoe100M dataset

In [ ]:
from pathlib import Path

import json

In [ ]:
PATH_ZARR = Path("/vol/data/annbatch_benchmark/tahoe100M/zarr_shuffled")
PATH_H5AD = Path("/vol/data/annbatch_benchmark/tahoe100M/h5ad_shuffled")

## Arrayloaders

In [ ]:
%%capture cap --no-stderr
!python annbatch/benchmark_annbatch.py --chunk_size=64 --preload_nchunks=128 --n_samples=2000000 --store_path={PATH_ZARR}

In [ ]:
arrayloaders_64 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_64

In [ ]:
%%capture cap --no-stderr
!python annbatch/benchmark_annbatch.py --chunk_size=128 --preload_nchunks=64 --n_samples=2000000 --store_path={PATH_ZARR}

In [ ]:
arrayloaders_128 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_128

In [ ]:
%%capture cap --no-stderr
!python annbatch/benchmark_annbatch.py --chunk_size=256 --preload_nchunks=32 --n_samples=2000000 --store_path={PATH_ZARR}

In [ ]:
arrayloaders_256 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_256

In [ ]:
%%capture cap --no-stderr
!python annbatch/benchmark_annbatch.py --chunk_size=512 --preload_nchunks=16 --n_samples=2000000 --store_path={PATH_ZARR}

In [ ]:
arrayloaders_512 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_512

In [ ]:
%%capture cap --no-stderr
!python annbatch/benchmark_annbatch.py --chunk_size=256 --preload_nchunks=32 --preload_to_gpu=True --n_samples=2000000 --store_path={PATH_ZARR}

In [ ]:
arrayloaders_gpu = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_gpu


## MappedCollection

In [ ]:
%%capture cap --no-stderr
!python mapped_collection/benchmark_mapped_collection.py --store_path={PATH_H5AD}

In [ ]:
mapped_collection = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
mapped_collection

## scDataset

In [ ]:
%%capture cap --no-stderr
!python scDataset/benchmark_scDataset.py --num_workers=8 --batch_size=64 --store_path={PATH_H5AD}  # recommended settings from tutorial

In [ ]:
sc_dataset = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset

In [ ]:
%%capture cap --no-stderr
!python scDataset/benchmark_scDataset.py --num_workers=8 --batch_size=4096 --block_size=256 --fetch_factor=2 --store_path={PATH_H5AD}

In [ ]:
sc_dataset_matched = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset_matched

## Plot results

In [ ]:
import pandas as pd

results = pd.DataFrame(
    {
        "loader": [
            "annbatch (chunk_size=64)",
            "annbatch (chunk_size=128)",
            "annbatch (chunk_size=256)",
            "annbatch (chunk_size=512)",
            "annbatch (GPU)",
            "MappedCollection",
            "scDataset (recommended settings)",
            "scDataset (annbatch matched settings)",
        ],
        "samples/sec": [
            arrayloaders_64,
            arrayloaders_128,
            arrayloaders_256,
            arrayloaders_512,
            arrayloaders_gpu,
            mapped_collection,
            sc_dataset,
            sc_dataset_matched,
        ],
    }
)
results.to_csv("loading_speed_results.csv", index=False)
results